In [0]:
%pip install -U mlflow transformers torch huggingface_hub sentencepiece accelerate 

In [0]:
import mlflow
import numpy as np
from sentence_transformers import SentenceTransformer
from mlflow.models.signature import ModelSignature
from mlflow.types import ColSpec, Schema, TensorSpec

# Download the model
model = SentenceTransformer(
    "LLMXperts/Arabic-Triplet-Matryoshka-V2"
)

# Get the embedding dimension
embedding_dim = model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {embedding_dim}")

# Define the signature explicitly
signature = ModelSignature(
    inputs=Schema([ColSpec(type="string", name="text")]),
    outputs=Schema([TensorSpec(type=np.dtype("float32"), shape=(-1, embedding_dim))])
)

# Log with signature
model_name = "marbert-matryoshka-embeddings"

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run():
    mlflow.sentence_transformers.log_model(
        model=model,
        artifact_path="model",
        registered_model_name=model_name,
        signature=signature,
        input_example=["مرحبا كيفك", "شو أخبارك"]
    )

In [0]:
model.save("/Volumes/amit/bertopic/models/marbert-matryoshka")


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

w.serving_endpoints.create_and_wait(
    name="marbert-matryoshka-embeddings",
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name="workspace.default.marbert-matryoshka-embeddings",
                entity_version="1",
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ]
    ),
)

In [0]:
%sql
SELECT ai_query(
    'marbert-matryoshka-embeddings',
    'كيف الحال يا شباب'
) AS embedding;

In [0]:
  loaded = mlflow.pyfunc.load_model(model_info.model_uri)
  loaded.predict(pd.DataFrame({"text": ["انت يا عميل السفارات يا ابن الكلب حسابك عسير"]}))

In [0]:
  import os
  import requests
  DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

  workspace_url = "https://dbc-de54b796-a6c4.cloud.databricks.com/"
  endpoint_name = "marbertv2-incitement-detector"
  token = DATABRICKS_TOKEN

  payload = {
      "dataframe_records": [
          {"text": "انت يا عميل السفارات يا ابن الكلب حسابك عسير"}
      ]
  }

  resp = requests.post(
      f"{workspace_url}/serving-endpoints/{endpoint_name}/invocations",
      headers={
          "Authorization": f"Bearer {token}",
          "Content-Type": "application/json",
      },
      json=payload,
      timeout=120,
  )

  print(resp.status_code)
  print(resp.json())

In [0]:
loaded.predict(pd.DataFrame({"text": ["انت يا عميل السفارات يا ابن الكلب حسابك عسير"]}))